In [1]:
%load_ext autoreload
%autoreload 2

import mlflow
from catboost import CatBoostRegressor
from src.data_prep import load_data
from src.feature_eng import features, cat_features
from src.train import train_model, log_experiment
from src.train import save_model
import sys
sys.path.insert(0, "/home/apc/dev/portfolio-ml/ml/examscore")


In [2]:
#Exam_Score_Prediction.csv
path = "data/Exam_Score_Prediction.csv"
df = load_data(path)

In [3]:
X = df[features]
y = df["exam_score"]

In [4]:
mlflow.set_experiment("catboost_baseline")
log_params = {
    "iterations": 700,
    "depth": 7,
    "learning_rate": 0.05,
}
unlog_params = {
    "random_state": 42,
    "verbose": False,
}
model_params = log_params | unlog_params

mode = "holdout"

with mlflow.start_run():

    model = CatBoostRegressor(**model_params, cat_features=cat_features)

    results, n_rows = train_model(
        model=model,
        X=X,
        y=y,
        mode=mode,
        test_size=0.2,
        random_state=42,
    )

    log_experiment(results, model, log_params, mode, features, path, n_rows)

    

Доступные метрики: ['rmse', 'nrmse', 'r2']
RMSE:  9.85
NRMSE: 15.79%
R2:    0.7286
dict_items([('iterations', 700), ('learning_rate', 0.05), ('depth', 7), ('loss_function', 'RMSE'), ('verbose', False), ('random_state', 42), ('cat_features', ['sleep_quality', 'study_method', 'facility_rating'])])


In [5]:
final_model = CatBoostRegressor(**model_params)
final_model.fit(X, y, cat_features=cat_features)

save_model(final_model, path="../../../backend/model-examscore.pkl")

Модель сохранена: /home/apc/dev/portfolio-ml/backend/model-examscore.pkl


In [6]:
import inspect
import src.train as train_module

print(train_module.__file__)
print(inspect.signature(train_module.save_model))

/home/apc/dev/portfolio-ml/ml/examscore/src/train.py
(model, path='model.pkl')


In [7]:
import os
print(os.getcwd())

/home/apc/dev/portfolio-ml/ml/examscore/notebooks
